In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 41 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (23.9 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package fonts-nanum.
(Reading database ... 125080 files and direc

#데이터 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rc('font', family='NanumBarunGothic')

!pip install tspoon
import tspoon as tsp

In [ ]:
!pip install xmltodict
import pandas as pd

import requests
import xml.etree.ElementTree as ET
import xml.dom.minidom
import xmltodict

import json
from urllib.request import urlopen

def getECOS(topic,period,beg,end,item1='?',item2='?',item3='?',item4='?', df=None):
    key = ''
    url = 'https://ecos.bok.or.kr/api/StatisticSearch/'+'OIT9OYFOZDZWLHSBQJWU'+'/xml/kr/'+str(1)+'/'+str(50000)+'/'+topic+'/'+period+'/'+beg+'/'+end \
            +'/'+item1+'/'+item2+'/'+item3+'/'+item4
    # print(url)
    ## call OpenAPI URL
    response = requests.get(url)

    ## get API return value upon the http request
    if response.status_code == 200:
        try:
            contents = response.text
            ecosRoot = ET.fromstring(contents)

            # error check
            if ecosRoot[0].text[:4] in ("INFO","ERRO"):
                print(ecosRoot[0].text + " : " + ecosRoot[1].text)

            else:
                #print(contents)
                #dom = xml.dom.minidom.parse(xml_fname)
                dom = xml.dom.minidom.parseString(contents)
                pretty_xml_as_string = dom.toprettyxml(indent=" ")
                # print(pretty_xml_as_string)
                dic = xmltodict.parse(pretty_xml_as_string)

                n = int(dic['StatisticSearch']['list_total_count']['#text'])
                df_ecos = pd.DataFrame(index = [dic['StatisticSearch']['row'][i]['TIME'] for i in range(n)])
                df_ecos[dic['StatisticSearch']['row'][0]['ITEM_NAME1']] = [float(dic['StatisticSearch']['row'][i]['DATA_VALUE']) for i in range(n)]

                if type(df)==type(pd.DataFrame()):
                    df_ecos = df.merge(df_ecos, left_index=True, right_index=True, how='left')

                return df_ecos

        except Exception as e:
            print(str(e))


def getKOSIS(table,period,beg,end,item, orgId='101', obj1='',obj2='',obj3='',obj4='',obj5='',obj6='',obj7='',obj8='', title=None, title_no='0', debug=False):
     # Korean Statistics
    key = 'MzhiMmQwZTYyMWZiNGM1ZGFkZDJmZTI2OTE1OGU2NzQ='
    url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey='+key \
            +'&orgId='+orgId\
            +'&tblId='+table \
            +'&prdSe='+period \
            +'&startPrdDe='+beg \
            +'&endPrdDe='+end \
            +'&itmId='+item \
            +'&objL1='+obj1 \
            +'&objL2='+obj2 \
            +'&objL3='+obj3 \
            +'&objL4='+obj4 \
            +'&objL5='+obj5 \
            +'&objL6='+obj6 \
            +'&objL7='+obj7 \
            +'&objL8='+obj8 \
            +'&format=json&jsonVD=Y&outputFields=ITM_NM+PRD_DE+DT'
    res = requests.get(url)
    py_json = res.json()

    data = []
    for v in py_json:
        value = []
        value.append(v['PRD_DE'])

        # ✅ DT 값이 '-' 또는 ''일 때 NaN으로 처리
        try:
            value.append(float(v['DT']))
        except (ValueError, KeyError):
            value.append(np.nan)

        data.append(value)

    df = pd.DataFrame(data, columns=['PRD_DE', 'DT'])

    if debug:
        print(df.head())

    return df


    #get json data from url
    with urlopen(url) as url_:
        json_file = url_.read()

    py_json = json.loads(json_file.decode('utf-8'))


    #data transformation
    data = []

    for i, v in enumerate(py_json):
        value = []
        value.append(v['PRD_DE'])
        value.append(float(v['DT']))

        data.append(value)

    if title is not None:
        title = str(title)
    elif title_no=='0':
        title = v['ITM_NM_ENG']
    else:
        title = v['C'+title_no+'_NM_ENG']

    df_kosis = pd.DataFrame({v['PRD_SE']:[x[0] for x in data], title:[x[1] for x in data]}).set_index(v['PRD_SE'])

    if debug:
        return v

    return df_kosis

In [ ]:
import pandas as pd
import numpy as np
import os
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Mounted at /content/gdrive/


In [ ]:
# change directory to where you have your data file!!
%cd gdrive/MyDrive/GDP/데이터

/content/gdrive/MyDrive/GDP/데이터


##GDP

In [ ]:
gdp = getECOS('200Y104', 'Q', '2014Q1', '2025Q2','110312').reset_index()
gdp

,index,운송장비 제조업
0,2014Q1,17740.3
1,2014Q2,17944.4
2,2014Q3,17202.3
3,2014Q4,16510.7
4,2015Q1,16388.1
5,2015Q2,17144.1
6,2015Q3,17627.4
7,2015Q4,17287.4
8,2016Q1,17222.6
9,2016Q2,17901.5


##PPI

In [ ]:
ppi = getECOS('404Y014','Q','2014Q1','2025Q2','312AA').reset_index()
ppi

,index,운송장비
0,2014Q1,93.83
1,2014Q2,95.66
2,2014Q3,97.20
3,2014Q4,97.70
4,2015Q1,97.94
5,2015Q2,97.88
6,2015Q3,97.89
7,2015Q4,97.64
8,2016Q1,97.83
9,2016Q2,97.75


In [ ]:
ppi = ppi.rename(columns={'운송장비':'ppi'})

##BSI(단순평균으로 계산)

In [ ]:
업황실적BSI1 = getECOS('512Y007','M','201401','202506','AA','C3000').reset_index()
업황실적BSI2 = getECOS('512Y007','M','201401','202506','AA','C3100').reset_index()

In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI1,업황실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
업황실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
업황실적BSI

업황실적BSI['업황실적BSI']=(업황실적BSI['업황실적BSI 1)_x']+업황실적BSI['업황실적BSI 1)_y'])/2
업황실적BSI = 업황실적BSI.drop(columns=["업황실적BSI 1)_x","업황실적BSI 1)_y"])
업황실적BSI

,index,업황실적BSI
0,201401,81.0
1,201402,87.0
2,201403,83.5
3,201404,87.0
4,201405,83.5
...,...,...
133,202502,87.0
134,202503,91.5
135,202504,89.5
136,202505,94.0


In [ ]:
매출실적BSI1 = getECOS('512Y007','M','201401','202506','AB','C3000').reset_index()
매출실적BSI2 = getECOS('512Y007','M','201401','202506','AB','C3100').reset_index()

from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [매출실적BSI1,매출실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
매출실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
매출실적BSI

매출실적BSI['매출실적BSI']=(매출실적BSI['매출실적BSI 2)_x']+매출실적BSI['매출실적BSI 2)_y'])/2
매출실적BSI = 매출실적BSI.drop(columns=["매출실적BSI 2)_x","매출실적BSI 2)_y"])
매출실적BSI

,index,매출실적BSI
0,201401,97.0
1,201402,89.0
2,201403,92.0
3,201404,101.5
4,201405,102.0
...,...,...
133,202502,93.0
134,202503,95.5
135,202504,101.0
136,202505,102.5


In [ ]:
채산성실적BSI1 = getECOS('512Y007','M','201401','202506','AE','C3000').reset_index()
채산성실적BSI2 = getECOS('512Y007','M','201401','202506','AE','C3100').reset_index()
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [채산성실적BSI1,채산성실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
채산성실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
채산성실적BSI

채산성실적BSI['채산성실적BSI']=(채산성실적BSI['채산성실적BSI 6)_x']+채산성실적BSI['채산성실적BSI 6)_y'])/2
채산성실적BSI = 채산성실적BSI.drop(columns=["채산성실적BSI 6)_x","채산성실적BSI 6)_y"])
채산성실적BSI

,index,채산성실적BSI
0,201401,97.0
1,201402,93.5
2,201403,94.5
3,201404,100.0
4,201405,97.0
...,...,...
133,202502,89.0
134,202503,94.5
135,202504,89.0
136,202505,90.5


In [ ]:
수출실적BSI1= getECOS('512Y007','M','201401','202506','AM','C3000').reset_index()
수출실적BSI2= getECOS('512Y007','M','201401','202506','AM','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [수출실적BSI1,수출실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
수출실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
수출실적BSI

수출실적BSI['수출실적BSI']=(수출실적BSI['수출실적BSI 2)_x']+수출실적BSI['수출실적BSI 2)_y'])/2
수출실적BSI = 수출실적BSI.drop(columns=["수출실적BSI 2)_x","수출실적BSI 2)_y"])
수출실적BSI

,index,수출실적BSI
0,201401,92.0
1,201402,88.0
2,201403,94.5
3,201404,102.0
4,201405,104.0
...,...,...
133,202502,99.5
134,202503,96.0
135,202504,100.5
136,202505,107.5


In [ ]:
내수판매실적BSI1= getECOS('512Y007','M','201401','202506','AL','C3000').reset_index()
내수판매실적BSI2= getECOS('512Y007','M','201401','202506','AL','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [내수판매실적BSI1,내수판매실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
내수판매실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
내수판매실적BSI


내수판매실적BSI['내수판매실적BSI']=(내수판매실적BSI['내수판매실적BSI 2)_x']+내수판매실적BSI['내수판매실적BSI 2)_y'])/2
내수판매실적BSI = 내수판매실적BSI.drop(columns=["내수판매실적BSI 2)_x","내수판매실적BSI 2)_y"])
내수판매실적BSI

,index,내수판매실적BSI
0,201401,87.5
1,201402,87.0
2,201403,89.0
3,201404,94.0
4,201405,95.0
...,...,...
133,202502,89.5
134,202503,91.5
135,202504,100.0
136,202505,95.5


In [ ]:
생산실적BSI1= getECOS('512Y007','M','201401','202506','AC','C3000').reset_index()
생산실적BSI2= getECOS('512Y007','M','201401','202506','AC','C3100').reset_index()
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [생산실적BSI1,생산실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
생산실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
생산실적BSI
생산실적BSI['생산실적BSI']=(생산실적BSI['생산실적BSI 2)_x']+생산실적BSI['생산실적BSI 2)_y'])/2
생산실적BSI = 생산실적BSI.drop(columns=["생산실적BSI 2)_x","생산실적BSI 2)_y"])
생산실적BSI

,index,생산실적BSI
0,201401,94.5
1,201402,90.5
2,201403,90.0
3,201404,104.0
4,201405,106.0
...,...,...
133,202502,92.0
134,202503,101.5
135,202504,97.0
136,202505,98.5


In [ ]:
신규수주실적BSI1= getECOS('512Y007','M','201401','202506','AD','C3000').reset_index()
신규수주실적BSI2= getECOS('512Y007','M','201401','202506','AD','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [신규수주실적BSI1,신규수주실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
신규수주실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
신규수주실적BSI

신규수주실적BSI['신규수주실적BSI']=(신규수주실적BSI['신규수주실적BSI 2)_x']+신규수주실적BSI['신규수주실적BSI 2)_y'])/2
신규수주실적BSI = 신규수주실적BSI.drop(columns=["신규수주실적BSI 2)_x","신규수주실적BSI 2)_y"])
신규수주실적BSI

,index,신규수주실적BSI
0,201401,97.5
1,201402,95.5
2,201403,95.0
3,201404,95.5
4,201405,94.0
...,...,...
133,202502,83.0
134,202503,86.0
135,202504,84.0
136,202505,88.5


In [ ]:

제품재고실적BSI1= getECOS('512Y007','M','201401','202506','AG','C3000').reset_index()
제품재고실적BSI2= getECOS('512Y007','M','201401','202506','AG','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [제품재고실적BSI1,제품재고실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
제품재고실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
제품재고실적BSI


제품재고실적BSI['제품재고실적BSI']=(제품재고실적BSI['제품재고실적BSI 3)_x']+제품재고실적BSI['제품재고실적BSI 3)_y'])/2
제품재고실적BSI = 제품재고실적BSI.drop(columns=["제품재고실적BSI 3)_x","제품재고실적BSI 3)_y"])
제품재고실적BSI

,index,제품재고실적BSI
0,201401,102.5
1,201402,102.5
2,201403,100.5
3,201404,102.5
4,201405,100.0
...,...,...
133,202502,103.0
134,202503,98.0
135,202504,101.0
136,202505,99.0


In [ ]:
가동률실적BSI1= getECOS('512Y007','M','201401','202506','AK','C3000').reset_index()
가동률실적BSI2= getECOS('512Y007','M','201401','202506','AK','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [가동률실적BSI1,가동률실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
가동률실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
가동률실적BSI

가동률실적BSI['가동률실적BSI']=(가동률실적BSI['가동률실적BSI 4)_x']+가동률실적BSI['가동률실적BSI 4)_y'])/2
가동률실적BSI = 가동률실적BSI.drop(columns=["가동률실적BSI 4)_x","가동률실적BSI 4)_y"])
가동률실적BSI

,index,가동률실적BSI
0,201401,95.0
1,201402,93.0
2,201403,94.0
3,201404,102.0
4,201405,98.5
...,...,...
133,202502,89.0
134,202503,99.5
135,202504,94.5
136,202505,96.5


In [ ]:
생산설비실적BSI1= getECOS('512Y007','M','201401','202506','AH','C3000').reset_index()
생산설비실적BSI2= getECOS('512Y007','M','201401','202506','AH','C3100').reset_index()
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [생산설비실적BSI1,생산설비실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
생산설비실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
생산설비실적BSI

생산설비실적BSI['생산설비실적BSI']=(생산설비실적BSI['생산설비실적BSI 3)_x']+생산설비실적BSI['생산설비실적BSI 3)_y'])/2
생산설비실적BSI = 생산설비실적BSI.drop(columns=["생산설비실적BSI 3)_x","생산설비실적BSI 3)_y"])
생산설비실적BSI

,index,생산설비실적BSI
0,201401,103.0
1,201402,106.5
2,201403,104.5
3,201404,104.0
4,201405,101.5
...,...,...
120,202401,99.5
121,202402,100.5
122,202403,97.0
123,202404,96.5


In [ ]:
설비투자실적BSI1= getECOS('512Y007','M','201401','202506','AI','C3000').reset_index()
설비투자실적BSI2= getECOS('512Y007','M','201401','202506','AI','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [설비투자실적BSI1,설비투자실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
설비투자실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
설비투자실적BSI


설비투자실적BSI['설비투자실적BSI']=(설비투자실적BSI['설비투자실적BSI 5)_x']+설비투자실적BSI['설비투자실적BSI 5)_y'])/2
설비투자실적BSI = 설비투자실적BSI.drop(columns=["설비투자실적BSI 5)_x","설비투자실적BSI 5)_y"])
설비투자실적BSI

,index,설비투자실적BSI
0,201401,96.5
1,201402,92.5
2,201403,97.0
3,201404,94.0
4,201405,96.0
...,...,...
133,202502,94.5
134,202503,97.0
135,202504,96.0
136,202505,101.0


In [ ]:
자금사정실적BSI1 = getECOS('512Y007','M','201401','202506','AO','C3000').reset_index()
자금사정실적BSI2 = getECOS('512Y007','M','201401','202506','AO','C3100').reset_index()


# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [자금사정실적BSI1,자금사정실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
자금사정실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
자금사정실적BSI


자금사정실적BSI['자금사정실적BSI']=(자금사정실적BSI['자금사정실적BSI 6)_x']+자금사정실적BSI['자금사정실적BSI 6)_y'])/2
자금사정실적BSI = 자금사정실적BSI.drop(columns=["자금사정실적BSI 6)_x","자금사정실적BSI 6)_y"])
자금사정실적BSI

,index,자금사정실적BSI
0,201401,87.5
1,201402,90.0
2,201403,90.5
3,201404,92.5
4,201405,89.5
...,...,...
133,202502,92.5
134,202503,88.5
135,202504,94.5
136,202505,83.5


In [ ]:
인력사정실적BSI1 = getECOS('512Y007','M','201401','202506','AJ','C3000').reset_index()
인력사정실적BSI2 = getECOS('512Y007','M','201401','202506','AJ','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [인력사정실적BSI1,인력사정실적BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
인력사정실적BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
인력사정실적BSI

인력사정실적BSI['인력사정실적BSI']=(인력사정실적BSI['인력사정실적BSI 3)_x']+인력사정실적BSI['인력사정실적BSI 3)_y'])/2
인력사정실적BSI = 인력사정실적BSI.drop(columns=["인력사정실적BSI 3)_x","인력사정실적BSI 3)_y"])
인력사정실적BSI


,index,인력사정실적BSI
0,201401,101.5
1,201402,96.0
2,201403,91.0
3,201404,90.5
4,201405,91.5
...,...,...
133,202502,94.5
134,202503,87.5
135,202504,88.5
136,202505,91.5


In [ ]:
원자재구입가격BSI1 = getECOS('512Y007','M','201401','202506','AN','C3000').reset_index()
원자재구입가격BSI2 = getECOS('512Y007','M','201401','202506','AN','C3100').reset_index()

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [원자재구입가격BSI1,원자재구입가격BSI2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
원자재구입가격BSI = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
원자재구입가격BSI

원자재구입가격BSI['원자재구입가격BSI']=(원자재구입가격BSI['원자재구입가격실적BSI 4)_x']+원자재구입가격BSI['원자재구입가격실적BSI 4)_y'])/2
원자재구입가격BSI = 원자재구입가격BSI.drop(columns=["원자재구입가격실적BSI 4)_x","원자재구입가격실적BSI 4)_y"])
원자재구입가격BSI


,index,원자재구입가격BSI
0,201401,99.0
1,201402,102.0
2,201403,98.0
3,201404,106.5
4,201405,99.0
...,...,...
133,202502,119.5
134,202503,123.0
135,202504,122.5
136,202505,121.0


In [ ]:
from functools import reduce

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [업황실적BSI, 매출실적BSI,수출실적BSI, 내수판매실적BSI, 생산실적BSI, 신규수주실적BSI, 제품재고실적BSI, 가동률실적BSI,설비투자실적BSI, 채산성실적BSI, 자금사정실적BSI, 인력사정실적BSI,원자재구입가격BSI]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
df_merged = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
df_merged

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,원자재구입가격BSI
0,201401,81.0,97.0,92.0,87.5,94.5,97.5,102.5,95.0,96.5,97.0,87.5,101.5,99.0
1,201402,87.0,89.0,88.0,87.0,90.5,95.5,102.5,93.0,92.5,93.5,90.0,96.0,102.0
2,201403,83.5,92.0,94.5,89.0,90.0,95.0,100.5,94.0,97.0,94.5,90.5,91.0,98.0
3,201404,87.0,101.5,102.0,94.0,104.0,95.5,102.5,102.0,94.0,100.0,92.5,90.5,106.5
4,201405,83.5,102.0,104.0,95.0,106.0,94.0,100.0,98.5,96.0,97.0,89.5,91.5,99.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,202502,87.0,93.0,99.5,89.5,92.0,83.0,103.0,89.0,94.5,89.0,92.5,94.5,119.5
134,202503,91.5,95.5,96.0,91.5,101.5,86.0,98.0,99.5,97.0,94.5,88.5,87.5,123.0
135,202504,89.5,101.0,100.5,100.0,97.0,84.0,101.0,94.5,96.0,89.0,94.5,88.5,122.5
136,202505,94.0,102.5,107.5,95.5,98.5,88.5,99.0,96.5,101.0,90.5,83.5,91.5,121.0


In [ ]:
df_merged.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
수출실적BSI,0
내수판매실적BSI,0
생산실적BSI,0
신규수주실적BSI,0
제품재고실적BSI,0
가동률실적BSI,0
설비투자실적BSI,0


In [ ]:
import pandas as pd

# index가 숫자라면 문자열로 변환
df_merged['index'] = df_merged['index'].astype(str)

# 연도와 월 분리
df_merged['연도'] = df_merged['index'].str[:4].astype(int)
df_merged['월'] = df_merged['index'].str[4:6].astype(int)

# 월 → 분기 변환 함수
def month_to_quarter(month):
    if month in [1, 2, 3]:
        return 'Q1'
    elif month in [4, 5, 6]:
        return 'Q2'
    elif month in [7, 8, 9]:
        return 'Q3'
    else:
        return 'Q4'

# 분기 계산
df_merged['분기'] = df_merged['월'].apply(month_to_quarter)

# 분기 인덱스 생성 (예: 2015Q1)
df_merged['index'] = df_merged['연도'].astype(str) + df_merged['분기']

# 분기별 평균 집계
df_quarter = df_merged.groupby('index').mean(numeric_only=True).reset_index()

# 결과 확인
print(df_quarter.head())

    index    업황실적BSI     매출실적BSI     수출실적BSI  내수판매실적BSI     생산실적BSI  \
0  2014Q1  83.833333   92.666667   91.500000  87.833333   91.666667   
1  2014Q2  85.833333  102.000000  100.000000  95.166667  103.500000   
2  2014Q3  76.333333   86.500000   88.666667  83.666667   91.500000   
3  2014Q4  74.166667   93.833333   91.833333  89.666667   99.166667   
4  2015Q1  73.000000   87.666667   88.333333  86.500000   94.500000   

   신규수주실적BSI   제품재고실적BSI   가동률실적BSI  설비투자실적BSI   채산성실적BSI  자금사정실적BSI  \
0  96.000000  101.833333  94.000000  95.333333  95.000000  89.333333   
1  96.000000  102.833333  99.666667  96.166667  99.000000  91.333333   
2  85.500000  101.833333  90.166667  95.666667  93.166667  86.833333   
3  93.666667  102.500000  97.500000  95.333333  94.500000  86.500000   
4  83.666667  103.666667  91.666667  95.166667  95.000000  85.666667   

    인력사정실적BSI  원자재구입가격BSI      연도     월  
0   96.166667   99.666667  2014.0   2.0  
1   90.333333  101.333333  2014.0   5.0  
2   91.166667 

In [ ]:
df_quarter = df_quarter.drop(columns=['연도', '월'], errors='ignore')

In [ ]:
dfs = [ppi, gdp]
df_final = df_quarter.copy()
key_col = 'index'
for df in dfs:
    df_final = df_final.merge(df, on=key_col, how='left')

df_final

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,채산성실적BSI,자금사정실적BSI,인력사정실적BSI,원자재구입가격BSI,ppi,운송장비 제조업
0,2014Q1,83.833333,92.666667,91.500000,87.833333,91.666667,96.000000,101.833333,94.000000,95.333333,95.000000,89.333333,96.166667,99.666667,93.83,17740.3
1,2014Q2,85.833333,102.000000,100.000000,95.166667,103.500000,96.000000,102.833333,99.666667,96.166667,99.000000,91.333333,90.333333,101.333333,95.66,17944.4
2,2014Q3,76.333333,86.500000,88.666667,83.666667,91.500000,85.500000,101.833333,90.166667,95.666667,93.166667,86.833333,91.166667,101.500000,97.20,17202.3
3,2014Q4,74.166667,93.833333,91.833333,89.666667,99.166667,93.666667,102.500000,97.500000,95.333333,94.500000,86.500000,91.166667,100.666667,97.70,16510.7
4,2015Q1,73.000000,87.666667,88.333333,86.500000,94.500000,83.666667,103.666667,91.666667,95.166667,95.000000,85.666667,101.166667,96.500000,97.94,16388.1
5,2015Q2,74.833333,92.666667,92.500000,89.500000,96.833333,84.666667,103.833333,95.333333,94.666667,96.833333,87.166667,96.333333,98.000000,97.88,17144.1
6,2015Q3,64.166667,85.833333,87.333333,82.833333,93.000000,80.166667,104.000000,90.333333,91.000000,92.500000,82.333333,95.666667,103.833333,97.89,17627.4
7,2015Q4,73.333333,87.333333,87.000000,86.833333,95.166667,80.833333,98.666667,92.500000,95.833333,92.333333,81.833333,96.666667,99.500000,97.64,17287.4
8,2016Q1,65.166667,82.500000,84.166667,82.666667,88.166667,77.000000,103.833333,86.166667,91.666667,86.166667,79.166667,95.833333,98.666667,97.83,17222.6
9,2016Q2,62.833333,80.500000,89.333333,76.000000,87.166667,68.666667,97.166667,85.833333,88.833333,89.833333,78.000000,97.000000,102.833333,97.75,17901.5


In [ ]:
df_final.to_csv("운송장비1차.csv")

## 생산지수

In [ ]:
생산지수1 = getKOSIS('DT_1F02001','Q','201401','202502','T20','101','00','C30')
생산지수2 = getKOSIS('DT_1F02001','Q','201401','202502','T20','101','00','C31')

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [생산지수1,생산지수2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
생산지수 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
생산지수
생산지수['생산지수']=(생산지수['DT_x']+생산지수['DT_y'])/2
생산지수 = 생산지수.drop(columns=["DT_x","DT_y"])
생산지수


def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산지수['index']=생산지수['PRD_DE'].apply(to_quarter)
생산지수 = 생산지수.drop(columns=["PRD_DE"])
생산지수 = 생산지수.rename(columns={'DT': '생산지수'})
생산지수

,생산지수,index
0,141.9175,2014Q1
1,139.0690,2014Q2
2,134.2330,2014Q3
3,132.9790,2014Q4
4,135.2305,2015Q1
5,131.1905,2015Q2
6,130.0310,2015Q3
7,125.2675,2015Q4
8,126.1460,2016Q1
9,123.4555,2016Q2


https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T20+&objL1=C30+C31+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F02016

## 내수/수출출하지수

In [ ]:
수출출하지수1 = getKOSIS('DT_1F02016','Q','201401','202502','T21','101','C30')
수출출하지수2 = getKOSIS('DT_1F02016','Q','201401','202502','T21','101','C31')


# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [수출출하지수1,수출출하지수2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
수출출하지수 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
수출출하지수
수출출하지수['수출출하지수']=(수출출하지수['DT_x']+수출출하지수['DT_y'])/2
수출출하지수 = 수출출하지수.drop(columns=["DT_x","DT_y"])
수출출하지수

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

수출출하지수['index']=수출출하지수['PRD_DE'].apply(to_quarter)
수출출하지수 = 수출출하지수.drop(columns=["PRD_DE"])
수출출하지수 = 수출출하지수.rename(columns={'DT': '수출출하지수'})
수출출하지수

,수출출하지수,index
0,155.9725,2014Q1
1,154.0005,2014Q2
2,150.3085,2014Q3
3,151.3075,2014Q4
4,153.2565,2015Q1
5,149.4950,2015Q2
6,144.8730,2015Q3
7,138.8920,2015Q4
8,138.8810,2016Q1
9,135.2145,2016Q2


In [ ]:
내수출하지수1 = getKOSIS('DT_1F02016','Q','201401','202502','T20','101','C30')
내수출하지수2 = getKOSIS('DT_1F02016','Q','201401','202502','T20','101','C31')


# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [내수출하지수1,내수출하지수2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
내수출하지수 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
내수출하지수
내수출하지수['내수출하지수']=(내수출하지수['DT_x']+내수출하지수['DT_y'])/2
내수출하지수 = 내수출하지수.drop(columns=["DT_x","DT_y"])
내수출하지수

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

내수출하지수['index']=내수출하지수['PRD_DE'].apply(to_quarter)
내수출하지수 = 내수출하지수.drop(columns=["PRD_DE"])
내수출하지수 = 내수출하지수.rename(columns={'DT': '내수출하지수'})
내수출하지수

,내수출하지수,index
0,86.7405,2014Q1
1,76.6745,2014Q2
2,69.9075,2014Q3
3,66.3610,2014Q4
4,64.2700,2015Q1
5,65.8325,2015Q2
6,73.1780,2015Q3
7,77.0180,2015Q4
8,77.7615,2016Q1
9,86.1450,2016Q2


##생산능력지수 &가동률지수

https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=MGNkMzY3MTdhNjZlMTNiZDMxYjU4NDA4NTdjZWU0YmQ=&itmId=T10+T20+&objL1=C30+C31+&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Q&startPrdDe=201401&endPrdDe=202502&outputFields=OBJ_NM+NM+ITM_NM+PRD_DE+&orgId=101&tblId=DT_1F32001

In [ ]:
생산능력지수1 = getKOSIS('DT_1F32001','Q','201401','202502','T10','101','C30')
생산능력지수2 = getKOSIS('DT_1F32001','Q','201401','202502','T10','101','C31')

# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [생산능력지수1,생산능력지수2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
생산능력지수 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
생산능력지수
생산능력지수['생산능력지수']=(생산능력지수['DT_x']+생산능력지수['DT_y'])/2
생산능력지수 = 생산능력지수.drop(columns=["DT_x","DT_y"])
생산능력지수

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

생산능력지수['index']=생산능력지수['PRD_DE'].apply(to_quarter)
생산능력지수 = 생산능력지수.drop(columns=["PRD_DE"])
생산능력지수 = 생산능력지수.rename(columns={'DT': '생산능력지수'})
생산능력지수

,생산능력지수,index
0,134.7225,2014Q1
1,135.1870,2014Q2
2,136.9580,2014Q3
3,137.5535,2014Q4
4,137.6245,2015Q1
5,136.2795,2015Q2
6,134.6695,2015Q3
7,134.6370,2015Q4
8,133.5430,2016Q1
9,131.5960,2016Q2


In [ ]:
가동률지수1 = getKOSIS('DT_1F32001','Q','201401','202502','T30','101','C30')
가동률지수2 = getKOSIS('DT_1F32001','Q','201401','202502','T30','101','C31')
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [가동률지수1,가동률지수2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
가동률지수 = reduce(lambda left, right: pd.merge(left, right, on="PRD_DE", how="outer"), dfs)
가동률지수
가동률지수['가동률지수']=(가동률지수['DT_x']+가동률지수['DT_y'])/2
가동률지수 = 가동률지수.drop(columns=["DT_x","DT_y"])
가동률지수

def to_quarter(x):
    year = str(x)[:4]   # 앞 4자리 → 연도
    q = str(x)[-2:]     # 뒤 2자리 → 분기
    return f"{year}Q{q[-1]}"  # 예: 200904 → 2009Q4

가동률지수['index']=가동률지수['PRD_DE'].apply(to_quarter)
가동률지수 = 가동률지수.drop(columns=["PRD_DE"])
가동률지수 = 가동률지수.rename(columns={'DT': '가동률지수'})
가동률지수

,가동률지수,index
0,109.8195,2014Q1
1,107.8225,2014Q2
2,102.6310,2014Q3
3,103.4900,2014Q4
4,111.5095,2015Q1
5,107.1530,2015Q2
6,107.9690,2015Q3
7,102.5900,2015Q4
8,103.5050,2016Q1
9,100.2165,2016Q2


##자동차

In [ ]:
자동차 = pd.read_csv('자동차.csv')

In [ ]:
자동차
자동차 = 자동차.drop(columns=["Unnamed: 0"])

In [ ]:
자동차

,index,KGM,kia
0,2014Q1,36671.0,432531
1,2014Q2,37564.0,444863
2,2014Q3,32012.0,390752
3,2014Q4,34800.0,438493
4,2015Q1,32915.0,410613
5,2015Q2,36885.0,454259
6,2015Q3,34074.0,386460
7,2015Q4,40890.0,474394
8,2016Q1,33666.0,384979
9,2016Q2,40911.0,404590


In [ ]:
자동차['자동차']=자동차['KGM']+자동차['kia']
자동차 = 자동차.drop(columns=["KGM","kia"])
자동차

,index,자동차
0,2014Q1,469202.0
1,2014Q2,482427.0
2,2014Q3,422764.0
3,2014Q4,473293.0
4,2015Q1,443528.0
5,2015Q2,491144.0
6,2015Q3,420534.0
7,2015Q4,515284.0
8,2016Q1,418645.0
9,2016Q2,445501.0


##기업데이터

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('임지오기업.csv')

In [ ]:
운송장비기업1 = df[df['매핑한 산업'] == "운송장비 제조업_자동차"]
운송장비기업2 = df[df['매핑한 산업'] == "운송장비 제조업_조선기타운수"]

In [ ]:
columns = ['EBITDA', '인건비']
grouped1 = 운송장비기업1.groupby('조사일')[columns].sum()
grouped1

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,2.871476e+08
2000-04-01,0.000000e+00,4.273023e+08
2000-07-01,0.000000e+00,2.268228e+08
2000-10-01,3.597134e+12,3.581286e+09
2001-01-01,0.000000e+00,2.983169e+08
...,...,...
2024-04-01,7.586095e+12,1.315979e+09
2024-07-01,5.126818e+12,1.523017e+09
2024-10-01,5.292829e+12,2.798115e+09


In [ ]:
grouped1 = grouped1.reset_index()

In [ ]:
grouped1['조사일']=pd.to_datetime(grouped1['조사일'])

# 분기(Quarter) 단위로 변환
grouped1["index"] = grouped1["조사일"].dt.to_period("Q")
grouped1 = grouped1.drop(columns=["조사일"])
print(grouped1)

grouped1

           EBITDA           인건비   index
0    0.000000e+00  2.871476e+08  2000Q1
1    0.000000e+00  4.273023e+08  2000Q2
2    0.000000e+00  2.268228e+08  2000Q3
3    3.597134e+12  3.581286e+09  2000Q4
4    0.000000e+00  2.983169e+08  2001Q1
..            ...           ...     ...
97   7.586095e+12  1.315979e+09  2024Q2
98   5.126818e+12  1.523017e+09  2024Q3
99   5.292829e+12  2.798115e+09  2024Q4
100  6.110609e+12  1.280934e+09  2025Q1
101  6.257252e+12 -1.279113e+09  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,2.871476e+08,2000Q1
1,0.000000e+00,4.273023e+08,2000Q2
2,0.000000e+00,2.268228e+08,2000Q3
3,3.597134e+12,3.581286e+09,2000Q4
4,0.000000e+00,2.983169e+08,2001Q1
...,...,...,...
97,7.586095e+12,1.315979e+09,2024Q2
98,5.126818e+12,1.523017e+09,2024Q3
99,5.292829e+12,2.798115e+09,2024Q4
100,6.110609e+12,1.280934e+09,2025Q1


In [ ]:
mask = (grouped1["index"] >= pd.Period("2014Q1")) & (grouped1["index"] <= pd.Period("2025Q2"))
기업1= grouped1.loc[mask]
기업1
기업1['합산'] = 기업1['EBITDA']+기업1['인건비']
기업1

기업1 = 기업1.rename(columns={'EBITDA': 'EBITDA1'})
기업1 = 기업1.rename(columns={'인건비': '인건비1'})
기업1 = 기업1.rename(columns={'합산': '합산1'})

/tmp/ipython-input-2907447296.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업1['합산'] = 기업1['EBITDA']+기업1['인건비']


In [ ]:
columns = ['EBITDA', '인건비']
grouped2 = 운송장비기업2.groupby('조사일')[columns].sum()
grouped2

,EBITDA,인건비
조사일,,
2000-01-01,0.000000e+00,16826463.0
2000-04-01,0.000000e+00,5297307.0
2000-07-01,0.000000e+00,3647703.0
2000-10-01,7.867222e+11,723800341.0
2001-01-01,0.000000e+00,11942467.0
...,...,...
2024-04-01,9.298956e+11,232153430.0
2024-07-01,1.118315e+12,202916546.0
2024-10-01,1.926277e+12,-407930775.0


In [ ]:
grouped2 = grouped2.reset_index()

In [ ]:
grouped2['조사일']=pd.to_datetime(grouped2['조사일'])

# 분기(Quarter) 단위로 변환
grouped2["index"] = grouped2["조사일"].dt.to_period("Q")
grouped2 = grouped2.drop(columns=["조사일"])
print(grouped2)

grouped2

           EBITDA          인건비   index
0    0.000000e+00   16826463.0  2000Q1
1    0.000000e+00    5297307.0  2000Q2
2    0.000000e+00    3647703.0  2000Q3
3    7.867222e+11  723800341.0  2000Q4
4    0.000000e+00   11942467.0  2001Q1
..            ...          ...     ...
97   9.298956e+11  232153430.0  2024Q2
98   1.118315e+12  202916546.0  2024Q3
99   1.926277e+12 -407930775.0  2024Q4
100  1.440408e+12  131225257.0  2025Q1
101  1.780024e+12 -128677364.0  2025Q2

[102 rows x 3 columns]


,EBITDA,인건비,index
0,0.000000e+00,16826463.0,2000Q1
1,0.000000e+00,5297307.0,2000Q2
2,0.000000e+00,3647703.0,2000Q3
3,7.867222e+11,723800341.0,2000Q4
4,0.000000e+00,11942467.0,2001Q1
...,...,...,...
97,9.298956e+11,232153430.0,2024Q2
98,1.118315e+12,202916546.0,2024Q3
99,1.926277e+12,-407930775.0,2024Q4
100,1.440408e+12,131225257.0,2025Q1


In [ ]:
mask = (grouped2["index"] >= pd.Period("2014Q1")) & (grouped2["index"] <= pd.Period("2025Q2"))
기업2= grouped2.loc[mask]
기업2
기업2['합산'] = 기업2['EBITDA']+기업2['인건비']
기업2

기업2 = 기업2.rename(columns={'EBITDA': 'EBITDA2'})
기업2 = 기업2.rename(columns={'인건비': '인건비2'})
기업2 = 기업2.rename(columns={'합산': '합산2'})

/tmp/ipython-input-1747950282.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  기업2['합산'] = 기업2['EBITDA']+기업2['인건비']


In [ ]:
기업1

,EBITDA1,인건비1,index,합산1
56,3.129183e+12,9.043311e+08,2014Q1,3.130087e+12
57,3.244810e+12,9.099992e+08,2014Q2,3.245720e+12
58,1.938718e+12,9.361093e+08,2014Q3,1.939654e+12
59,3.547278e+12,2.473065e+09,2014Q4,3.549751e+12
60,3.098155e+12,9.797514e+08,2015Q1,3.099134e+12
61,3.702805e+12,9.898881e+08,2015Q2,3.703795e+12
62,3.077155e+12,1.002433e+09,2015Q3,3.078158e+12
63,4.167052e+12,2.548467e+09,2015Q4,4.169601e+12
64,3.575275e+12,9.941619e+08,2016Q1,3.576269e+12
65,3.746672e+12,9.862464e+08,2016Q2,3.747659e+12


In [ ]:
기업2

,EBITDA2,인건비2,index,합산2
56,-1.967986e+11,7.092543e+07,2014Q1,-1.967277e+11
57,2.004036e+11,1.347470e+08,2014Q2,2.005384e+11
58,-3.874877e+11,1.015454e+08,2014Q3,-3.873862e+11
59,1.334126e+11,2.239488e+08,2014Q4,1.336366e+11
60,1.881541e+11,1.116292e+08,2015Q1,1.882657e+11
61,-1.692743e+12,1.281790e+08,2015Q2,-1.692615e+12
62,1.074329e+11,7.789841e+07,2015Q3,1.075108e+11
63,-1.543518e+10,1.734331e+08,2015Q4,-1.526175e+10
64,2.451261e+11,1.007656e+08,2016Q1,2.452268e+11
65,-2.999220e+09,1.340899e+08,2016Q2,-2.865130e+09


In [ ]:
# 불러온 BSIDataFrame들을 리스트로 모으기
dfs = [기업1,기업2]
dfs = [df.drop(columns=[col for col in df.columns if "Unnamed" in col]) for df in dfs]

# '시점' 기준으로 병합
기업 = reduce(lambda left, right: pd.merge(left, right, on="index", how="outer"), dfs)
기업

기업['EBITDA']=기업['EBITDA1']+기업['EBITDA2']
기업['인건비']=기업['인건비1']+기업['인건비2']
기업['합산']=기업['합산1']+기업['합산2']
기업

,EBITDA1,인건비1,index,합산1,EBITDA2,인건비2,합산2,EBITDA,인건비,합산
0,3.129183e+12,9.043311e+08,2014Q1,3.130087e+12,-1.967986e+11,7.092543e+07,-1.967277e+11,2.932384e+12,9.752566e+08,2.933360e+12
1,3.244810e+12,9.099992e+08,2014Q2,3.245720e+12,2.004036e+11,1.347470e+08,2.005384e+11,3.445213e+12,1.044746e+09,3.446258e+12
2,1.938718e+12,9.361093e+08,2014Q3,1.939654e+12,-3.874877e+11,1.015454e+08,-3.873862e+11,1.551231e+12,1.037655e+09,1.552268e+12
3,3.547278e+12,2.473065e+09,2014Q4,3.549751e+12,1.334126e+11,2.239488e+08,1.336366e+11,3.680691e+12,2.697014e+09,3.683388e+12
4,3.098155e+12,9.797514e+08,2015Q1,3.099134e+12,1.881541e+11,1.116292e+08,1.882657e+11,3.286309e+12,1.091381e+09,3.287400e+12
5,3.702805e+12,9.898881e+08,2015Q2,3.703795e+12,-1.692743e+12,1.281790e+08,-1.692615e+12,2.010062e+12,1.118067e+09,2.011180e+12
6,3.077155e+12,1.002433e+09,2015Q3,3.078158e+12,1.074329e+11,7.789841e+07,1.075108e+11,3.184588e+12,1.080332e+09,3.185668e+12
7,4.167052e+12,2.548467e+09,2015Q4,4.169601e+12,-1.543518e+10,1.734331e+08,-1.526175e+10,4.151617e+12,2.721900e+09,4.154339e+12
8,3.575275e+12,9.941619e+08,2016Q1,3.576269e+12,2.451261e+11,1.007656e+08,2.452268e+11,3.820401e+12,1.094928e+09,3.821496e+12
9,3.746672e+12,9.862464e+08,2016Q2,3.747659e+12,-2.999220e+09,1.340899e+08,-2.865130e+09,3.743673e+12,1.120336e+09,3.744794e+12


In [ ]:
기업 = 기업.drop(columns=["EBITDA1","EBITDA2","인건비1","인건비2","합산1","합산2"])
기업

,index,EBITDA,인건비,합산
0,2014Q1,2.932384e+12,9.752566e+08,2.933360e+12
1,2014Q2,3.445213e+12,1.044746e+09,3.446258e+12
2,2014Q3,1.551231e+12,1.037655e+09,1.552268e+12
3,2014Q4,3.680691e+12,2.697014e+09,3.683388e+12
4,2015Q1,3.286309e+12,1.091381e+09,3.287400e+12
5,2015Q2,2.010062e+12,1.118067e+09,2.011180e+12
6,2015Q3,3.184588e+12,1.080332e+09,3.185668e+12
7,2015Q4,4.151617e+12,2.721900e+09,4.154339e+12
8,2016Q1,3.820401e+12,1.094928e+09,3.821496e+12
9,2016Q2,3.743673e+12,1.120336e+09,3.744794e+12


##전체 데이터 합산

In [ ]:
from functools import reduce
import pandas as pd

# 모든 데이터프레임의 index 컬럼을 문자열로 변환
for df in [df_final,생산지수, 내수출하지수, 수출출하지수, 생산능력지수, 가동률지수,자동차,기업]:
    df['index'] = df['index'].astype(str)

# 리스트에 넣고 합치기
dfs = [df_final,생산지수, 내수출하지수, 수출출하지수, 생산능력지수, 가동률지수,자동차,기업]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='index', how='outer'), dfs)


In [ ]:
df_merged
df_merged = df_merged[['index'] + [col for col in df_merged.columns if col != 'index']]

In [ ]:
df_merged

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,운송장비 제조업,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,자동차,EBITDA,인건비,합산
0,2014Q1,83.833333,92.666667,91.500000,87.833333,91.666667,96.000000,101.833333,94.000000,95.333333,...,17740.3,141.9175,86.7405,155.9725,134.7225,109.8195,469202.0,2.932384e+12,9.752566e+08,2.933360e+12
1,2014Q2,85.833333,102.000000,100.000000,95.166667,103.500000,96.000000,102.833333,99.666667,96.166667,...,17944.4,139.0690,76.6745,154.0005,135.1870,107.8225,482427.0,3.445213e+12,1.044746e+09,3.446258e+12
2,2014Q3,76.333333,86.500000,88.666667,83.666667,91.500000,85.500000,101.833333,90.166667,95.666667,...,17202.3,134.2330,69.9075,150.3085,136.9580,102.6310,422764.0,1.551231e+12,1.037655e+09,1.552268e+12
3,2014Q4,74.166667,93.833333,91.833333,89.666667,99.166667,93.666667,102.500000,97.500000,95.333333,...,16510.7,132.9790,66.3610,151.3075,137.5535,103.4900,473293.0,3.680691e+12,2.697014e+09,3.683388e+12
4,2015Q1,73.000000,87.666667,88.333333,86.500000,94.500000,83.666667,103.666667,91.666667,95.166667,...,16388.1,135.2305,64.2700,153.2565,137.6245,111.5095,443528.0,3.286309e+12,1.091381e+09,3.287400e+12
5,2015Q2,74.833333,92.666667,92.500000,89.500000,96.833333,84.666667,103.833333,95.333333,94.666667,...,17144.1,131.1905,65.8325,149.4950,136.2795,107.1530,491144.0,2.010062e+12,1.118067e+09,2.011180e+12
6,2015Q3,64.166667,85.833333,87.333333,82.833333,93.000000,80.166667,104.000000,90.333333,91.000000,...,17627.4,130.0310,73.1780,144.8730,134.6695,107.9690,420534.0,3.184588e+12,1.080332e+09,3.185668e+12
7,2015Q4,73.333333,87.333333,87.000000,86.833333,95.166667,80.833333,98.666667,92.500000,95.833333,...,17287.4,125.2675,77.0180,138.8920,134.6370,102.5900,515284.0,4.151617e+12,2.721900e+09,4.154339e+12
8,2016Q1,65.166667,82.500000,84.166667,82.666667,88.166667,77.000000,103.833333,86.166667,91.666667,...,17222.6,126.1460,77.7615,138.8810,133.5430,103.5050,418645.0,3.820401e+12,1.094928e+09,3.821496e+12
9,2016Q2,62.833333,80.500000,89.333333,76.000000,87.166667,68.666667,97.166667,85.833333,88.833333,...,17901.5,123.4555,86.1450,135.2145,131.5960,100.2165,445501.0,3.743673e+12,1.120336e+09,3.744794e+12


In [ ]:
df_merged.to_csv('운송장비.csv')

#데이터QoQ반환

In [ ]:
import pandas as pd

# df는 위의 데이터프레임
df_qoq = df_merged.copy()

# 분기 순서대로 정렬 (혹시 index가 문자열일 경우를 대비)
df_qoq = df_qoq.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_qoq.select_dtypes(include=['number']).columns

# 전분기 대비 증감률 (%) 계산
df_qoq[num_cols] = df_qoq[num_cols].pct_change() * 100

# 결과 확인
print(df_qoq.head())


    index    업황실적BSI    매출실적BSI    수출실적BSI  내수판매실적BSI    생산실적BSI  신규수주실적BSI  \
0  2014Q1        NaN        NaN        NaN        NaN        NaN        NaN   
1  2014Q2   2.385686  10.071942   9.289617   8.349146  12.909091   0.000000   
2  2014Q3 -11.067961 -15.196078 -11.333333 -12.084063 -11.594203 -10.937500   
3  2014Q4  -2.838428   8.477842   3.571429   7.171315   8.378871   9.551657   
4  2015Q1  -1.573034  -6.571936  -3.811252  -3.531599  -4.705882 -10.676157   

   제품재고실적BSI  가동률실적BSI  설비투자실적BSI  ...  운송장비 제조업      생산지수     내수출하지수  \
0        NaN       NaN        NaN  ...       NaN       NaN        NaN   
1   0.981997  6.028369   0.874126  ...  1.150488 -2.007152 -11.604729   
2  -0.972447 -9.531773  -0.519931  ... -4.135552 -3.477410  -8.825620   
3   0.654664  8.133087  -0.348432  ... -4.020393 -0.934197  -5.073132   
4   1.138211 -5.982906  -0.174825  ... -0.742549  1.693124  -3.150947   

     수출출하지수    생산능력지수     가동률지수        자동차      EBITDA         인건비          합산  
0    

In [ ]:
df_qoq
df_qoq = df_qoq.drop(index=[0, 1, 2,3])

In [ ]:
df_qoq.isnull().sum()

,0
index,0
업황실적BSI,0
매출실적BSI,0
수출실적BSI,0
내수판매실적BSI,0
생산실적BSI,0
신규수주실적BSI,0
제품재고실적BSI,0
가동률실적BSI,0
설비투자실적BSI,0


In [ ]:
df_qoq.to_csv('운송장비qoq.csv')

#데이터YoY변환

In [ ]:
import pandas as pd

# df_merged는 원본 데이터프레임
df_yoy = df_merged.copy()

# 분기 순서대로 정렬
df_yoy = df_yoy.sort_values('index')

# 수치형 컬럼만 선택 (index 제외)
num_cols = df_yoy.select_dtypes(include=['number']).columns

# 전년 동분기 대비 증감률 (%) 계산 (4분기 차이)
df_yoy[num_cols] = df_yoy[num_cols].pct_change(periods=4) * 100

# 결과 확인
print(df_yoy.head(10))

    index    업황실적BSI    매출실적BSI   수출실적BSI  내수판매실적BSI   생산실적BSI  신규수주실적BSI  \
0  2014Q1        NaN        NaN       NaN        NaN       NaN        NaN   
1  2014Q2        NaN        NaN       NaN        NaN       NaN        NaN   
2  2014Q3        NaN        NaN       NaN        NaN       NaN        NaN   
3  2014Q4        NaN        NaN       NaN        NaN       NaN        NaN   
4  2015Q1 -12.922465  -5.395683 -3.460838  -1.518027  3.090909 -12.847222   
5  2015Q2 -12.815534  -9.150327 -7.500000  -5.954466 -6.441224 -11.805556   
6  2015Q3 -15.938865  -0.770713 -1.503759  -0.996016  1.639344  -6.237817   
7  2015Q4  -1.123596  -6.927176 -5.263158  -3.159851 -4.033613 -13.701068   
8  2016Q1 -10.730594  -5.893536 -4.716981  -4.431599 -6.701940  -7.968127   
9  2016Q2 -16.035635 -13.129496 -3.423423 -15.083799 -9.982788 -18.897638   

   제품재고실적BSI  가동률실적BSI  설비투자실적BSI  ...  운송장비 제조업      생산지수     내수출하지수  \
0        NaN       NaN        NaN  ...       NaN       NaN        NaN   
1     

In [ ]:
df_yoy=df_yoy.dropna()

In [ ]:
df_yoy

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,운송장비 제조업,생산지수,내수출하지수,수출출하지수,생산능력지수,가동률지수,자동차,EBITDA,인건비,합산
4,2015Q1,-12.922465,-5.395683,-3.460838,-1.518027,3.090909,-12.847222,1.800327,-2.482270,-0.174825,...,-7.622194,-4.711892,-25.905431,-1.741333,2.154057,1.538889,-5.471844,12.069501,11.907022,12.069447
5,2015Q2,-12.815534,-9.150327,-7.500000,-5.954466,-6.441224,-11.805556,0.972447,-4.347826,-1.559792,...,-4.459887,-5.665173,-14.140294,-2.925640,0.808140,-0.620928,1.806906,-41.656370,7.018053,-41.641614
6,2015Q3,-15.938865,-0.770713,-1.503759,-0.996016,1.639344,-6.237817,2.127660,0.184843,-4.878049,...,2.471181,-3.130378,4.678325,-3.616229,-1.670950,5.201158,-0.527481,105.294312,4.112831,105.226675
7,2015Q4,-1.123596,-6.927176,-5.263158,-3.159851,-4.033613,-13.701068,-3.739837,-5.128205,0.524476,...,4.704222,-5.799036,16.059131,-8.205476,-2.120266,-0.869649,8.872094,12.794512,0.922726,12.785819
8,2016Q1,-10.730594,-5.893536,-4.716981,-4.431599,-6.701940,-7.968127,0.160772,-6.000000,-3.677758,...,5.092110,-6.717789,20.991909,-9.380026,-2.965678,-7.178312,-5.610243,16.252052,0.325000,16.246764
9,2016Q2,-16.035635,-13.129496,-3.423423,-15.083799,-9.982788,-18.897638,-6.420546,-9.965035,-6.161972,...,4.417846,-5.896006,30.854821,-9.552493,-3.436687,-6.473454,-9.293201,86.246616,0.202964,86.198782
10,2016Q3,-14.285714,-19.611650,-7.061069,-22.132797,-17.562724,-21.205821,-7.532051,-18.819188,-3.479853,...,-3.921168,-10.793195,6.933095,-12.400861,-3.647819,-11.724199,-12.680544,-28.482767,0.543554,-28.472924
11,2016Q4,-23.181818,-20.229008,-12.068966,-19.577735,-18.388792,-12.164948,-3.209459,-16.036036,-7.304348,...,-1.146500,-7.697926,-5.884988,-6.710970,-4.298967,-6.405108,-7.379814,-9.956728,0.672642,-9.949764
12,2017Q1,-8.951407,-11.515152,-11.485149,-12.500000,-11.720227,-1.515152,-9.149278,-9.090909,-4.727273,...,-2.546073,-13.271130,-0.616629,-13.870868,-3.321777,-11.839525,-1.475713,-11.726809,-2.305986,-11.724109
13,2017Q2,-7.957560,-22.981366,-29.104478,-16.228070,-20.458891,9.466019,-3.602058,-20.000000,-1.125704,...,-8.721616,-10.275362,-0.438795,-13.838013,-2.750084,-8.979060,-5.367665,-9.551161,-6.888459,-9.550365


In [ ]:
df_yoy.to_csv("운송장비yoy.csv")

#상관계수 보기(qoq)

In [ ]:
qoq = pd.read_csv('운송장비qoq.csv')
qoq = qoq.drop(columns=["Unnamed: 0"])

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# GDP_QoQ 기준
target = '운송장비 제조업'

In [ ]:
# 수치형 컬럼만 선택
num_cols = qoq.select_dtypes(include=['number']).columns

# GDP 기준 상관계수 계산
corr = qoq[num_cols].corr()[target].sort_values(ascending=False)

print("📊 GDP 대비 단순 상관계수")
print(corr)


📊 GDP 대비 단순 상관계수
운송장비 제조업      1.000000
가동률실적BSI      0.618359
수출출하지수        0.617791
채산성실적BSI      0.583120
내수판매실적BSI     0.581421
생산실적BSI       0.579774
설비투자실적BSI     0.566483
업황실적BSI       0.564881
자금사정실적BSI     0.562443
생산지수          0.557835
매출실적BSI       0.534971
수출실적BSI       0.525605
가동률지수         0.497827
신규수주실적BSI     0.446430
자동차           0.368197
내수출하지수        0.234309
합산            0.078699
EBITDA        0.078631
생산능력지수        0.055524
인건비           0.027708
ppi          -0.061671
원자재구입가격BSI   -0.168317
제품재고실적BSI    -0.295202
인력사정실적BSI    -0.308234
Name: 운송장비 제조업, dtype: float64


In [ ]:
# df2 사본
df2 = qoq.copy()

# target 및 feature 지정
target_col = '운송장비 제조업'
feature_cols = [col for col in df2.columns if col not in ['index', target_col]]

# 결과 저장용 데이터프레임
corr_df = pd.DataFrame(columns=['Feature', 'Lag', 'Correlation'])

# lag 1~4 생성 및 상관계수 계산
for col in feature_cols:
    for lag in range(1, 5):  # lag 1~4
        lag_col_name = f"{col}_lag{lag}"
        df2[lag_col_name] = df2[col].shift(lag)  # 실제 데이터에 lag 컬럼 추가
        corr = df2[target_col].corr(df2[lag_col_name])  # 상관계수 계산
        corr_df = pd.concat(
            [corr_df, pd.DataFrame({'Feature': [col], 'Lag': [lag], 'Correlation': [corr]})],
            ignore_index=True
        )

# 상관계수 높은 순으로 정렬
corr_df.sort_values(by='Correlation', ascending=False, inplace=True)
corr_df.reset_index(drop=True, inplace=True)

corr_df

/tmp/ipython-input-92513580.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  corr_df = pd.concat(


,Feature,Lag,Correlation
0,ppi,2,0.367493
1,제품재고실적BSI,3,0.278509
2,생산능력지수,4,0.205837
3,EBITDA,3,0.198176
4,합산,3,0.198092
...,...,...,...
87,가동률실적BSI,3,-0.225568
88,자금사정실적BSI,3,-0.234805
89,업황실적BSI,3,-0.236985
90,내수판매실적BSI,3,-0.267055


In [ ]:
df2

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
0,2015Q1,-1.573034,-6.571936,-3.811252,-3.531599,-4.705882,-10.676157,1.138211,-5.982906,-0.174825,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015Q2,2.511416,5.703422,4.716981,3.468208,2.469136,1.195219,0.160772,4.000000,-0.525394,...,NaN,NaN,-59.533743,NaN,NaN,NaN,-10.750635,NaN,NaN,NaN
2,2015Q3,-14.253898,-7.374101,-5.585586,-7.448790,-3.958692,-5.314961,0.160514,-5.244755,-3.873239,...,NaN,NaN,2.445205,-59.533743,NaN,NaN,-38.821547,-10.750635,NaN,NaN
3,2015Q4,14.285714,1.747573,-0.381679,4.828974,2.329749,0.831601,-5.128205,2.398524,5.311355,...,-10.714889,NaN,-3.375049,2.445205,-59.533743,NaN,58.397933,-38.821547,-10.750635,NaN
4,2016Q1,-11.136364,-5.534351,-3.256705,-4.798464,-7.355517,-4.742268,5.236486,-6.846847,-4.347826,...,-38.835251,-10.714889,151.950379,-3.375049,2.445205,-59.533743,30.407137,58.397933,-38.821547,-10.750635
5,2016Q2,-3.580563,-2.424242,6.138614,-8.064516,-1.134216,-10.822511,-6.420546,-0.386847,-3.090909,...,58.432293,-38.835251,-59.773409,151.950379,-3.375049,2.445205,-8.011929,30.407137,58.397933,-38.821547
6,2016Q3,-12.466844,-14.285714,-9.141791,-15.131579,-12.045889,-8.009709,-1.029160,-14.563107,-1.125704,...,30.365905,58.432293,2.320589,-59.773409,151.950379,-3.375049,-2.007135,-8.011929,30.407137,58.397933
7,2016Q4,2.424242,0.966184,-5.749487,8.268734,1.304348,12.401055,-0.693241,5.909091,1.138520,...,-7.977993,30.365905,-3.046621,2.320589,-59.773409,151.950379,-39.152443,-2.007135,-8.011929,30.407137
8,2017Q1,5.325444,4.784689,-2.614379,3.579952,0.214592,6.807512,-1.221640,0.858369,-1.688555,...,-2.008375,-7.977993,152.273859,-3.046621,2.320589,-59.773409,64.178297,-39.152443,-2.007135,-8.011929
9,2017Q2,-2.528090,-15.068493,-14.988814,-11.981567,-10.920771,-0.879121,-0.706714,-12.340426,0.572519,...,-39.163248,-2.008375,-60.963603,152.273859,-3.046621,2.320589,-9.824458,64.178297,-39.152443,-2.007135


In [ ]:
# 숫자형 컬럼만 선택 (연도분기 제외)
numeric_df = df2.select_dtypes(include=["number"])

# 전체 상관계수
corr = numeric_df.corr()

target_corr = corr["운송장비 제조업"].drop("운송장비 제조업")
target_corr = target_corr.sort_values(ascending=False)
target_corr

,운송장비 제조업
가동률실적BSI,0.618359
수출출하지수,0.617791
채산성실적BSI,0.583120
내수판매실적BSI,0.581421
생산실적BSI,0.579774
...,...
업황실적BSI_lag3,-0.236985
내수판매실적BSI_lag3,-0.267055
채산성실적BSI_lag3,-0.292451
제품재고실적BSI,-0.295202


In [ ]:
target_corr.to_csv('운송장비순서.csv')

In [ ]:
df2.dropna(inplace=True)

In [ ]:
df2

,index,업황실적BSI,매출실적BSI,수출실적BSI,내수판매실적BSI,생산실적BSI,신규수주실적BSI,제품재고실적BSI,가동률실적BSI,설비투자실적BSI,...,EBITDA_lag3,EBITDA_lag4,인건비_lag1,인건비_lag2,인건비_lag3,인건비_lag4,합산_lag1,합산_lag2,합산_lag3,합산_lag4
4,2016Q1,-11.136364,-5.534351,-3.256705,-4.798464,-7.355517,-4.742268,5.236486,-6.846847,-4.347826,...,-38.835251,-10.714889,151.950379,-3.375049,2.445205,-59.533743,30.407137,58.397933,-38.821547,-10.750635
5,2016Q2,-3.580563,-2.424242,6.138614,-8.064516,-1.134216,-10.822511,-6.420546,-0.386847,-3.090909,...,58.432293,-38.835251,-59.773409,151.950379,-3.375049,2.445205,-8.011929,30.407137,58.397933,-38.821547
6,2016Q3,-12.466844,-14.285714,-9.141791,-15.131579,-12.045889,-8.009709,-1.029160,-14.563107,-1.125704,...,30.365905,58.432293,2.320589,-59.773409,151.950379,-3.375049,-2.007135,-8.011929,30.407137,58.397933
7,2016Q4,2.424242,0.966184,-5.749487,8.268734,1.304348,12.401055,-0.693241,5.909091,1.138520,...,-7.977993,30.365905,-3.046621,2.320589,-59.773409,151.950379,-39.152443,-2.007135,-8.011929,30.407137
8,2017Q1,5.325444,4.784689,-2.614379,3.579952,0.214592,6.807512,-1.221640,0.858369,-1.688555,...,-2.008375,-7.977993,152.273859,-3.046621,2.320589,-59.773409,64.178297,-39.152443,-2.007135,-8.011929
9,2017Q2,-2.528090,-15.068493,-14.988814,-11.981567,-10.920771,-0.879121,-0.706714,-12.340426,0.572519,...,-39.163248,-2.008375,-60.963603,152.273859,-3.046621,2.320589,-9.824458,64.178297,-39.152443,-2.007135
10,2017Q3,-7.492795,-11.021505,-11.578947,-3.403141,-5.528846,-1.773836,3.024911,-4.611650,-5.313093,...,64.136282,-39.163248,-2.478900,-60.963603,152.273859,-3.046621,0.405885,-9.824458,64.178297,-39.152443
11,2017Q4,4.984424,6.042296,10.416667,-2.439024,5.089059,4.063205,-3.799655,2.544529,-0.200401,...,-9.786972,64.136282,8.345651,-2.478900,-60.963603,152.273859,-54.380753,0.405885,-9.824458,64.178297
12,2018Q1,4.747774,0.854701,1.078167,1.388889,-0.242131,1.084599,8.078995,3.970223,5.421687,...,0.406800,-9.786972,176.233030,8.345651,-2.478900,-60.963603,4.825299,-54.380753,0.405885,-9.824458
13,2018Q2,2.549575,12.146893,11.200000,10.136986,5.582524,-2.575107,-0.830565,8.353222,0.000000,...,-54.400077,0.406800,-67.302833,176.233030,8.345651,-2.478900,15.244750,4.825299,-54.380753,0.405885


In [ ]:
df2 = df2.reset_index()
df2 = df2.drop(columns=["level_0"])

In [ ]:
df2.to_csv('운송장비lag.csv', encoding='utf-8-sig')

#ARIMA

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운송장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '운송장비 제조업'   # GDP 예측 대상 칼럼
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI','설비투자실적BSI','업황실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (3.0, 1.0, 0.0)
📉 평균 Train MAE: 3.5676
📈 평균 Test MAE: 3.8985


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv('운송장비lag.csv')

# 분기형 인덱스로 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X) 설정
# -----------------------------
target = '운송장비 제조업'   # GDP 예측 대상 칼럼
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI']
y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X의 인덱스 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 예측 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

# 예측 구간: 2017Q1 이후부터 2025Q2까지
start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 파라미터 범위 설정
# -----------------------------
p_values = range(0, 4)
d_values = range(1, 4)
q_values = range(0, 4)

# -----------------------------
# 5️⃣ 결과 저장
# -----------------------------
results = []

# -----------------------------
# 6️⃣ (p,d,q) 조합별 탐색
# -----------------------------
for p in p_values:
    for d in d_values:
        for q in q_values:

            train_mae_list = []
            test_mae_list = []

            for test_start in pd.period_range(start_period, end_period, freq='Q'):
                train_end = test_start - 1
                test_end = test_start
                train_start = train_end - (train_window - 1)

                train_data = y.loc[train_start:train_end]
                test_data = y.loc[test_end:test_end]
                X_train = X.loc[train_start:train_end]
                X_test = X.loc[test_end:test_end]

                if len(train_data) < train_window or len(X_test) == 0:
                    continue

                try:
                    model = SARIMAX(train_data, exog=X_train, order=(p, d, q),
                                    enforce_stationarity=False, enforce_invertibility=False)
                    fit = model.fit(disp=False)

                    # 예측
                    pred = fit.forecast(steps=1, exog=X_test)

                    # Train MAE (fittedvalues vs train_data)
                    train_mae = mean_absolute_error(train_data, fit.fittedvalues)
                    train_mae_list.append(train_mae)

                    # Test MAE (out-of-sample)
                    test_mae = mean_absolute_error(test_data, pred)
                    test_mae_list.append(test_mae)

                except Exception:
                    continue

            if train_mae_list and test_mae_list:
                results.append({
                    'p': p, 'd': d, 'q': q,
                    'train_MAE': np.mean(train_mae_list),
                    'test_MAE': np.mean(test_mae_list)
                })

# -----------------------------
# 7️⃣ 결과 정리 및 최적 모델 출력
# -----------------------------
results_df = pd.DataFrame(results)
best = results_df.loc[results_df['test_MAE'].idxmin()]

print("✅ 최적 ARIMAX(p,d,q):", tuple(best[['p', 'd', 'q']]))
print("📉 평균 Train MAE:", round(best['train_MAE'], 4))
print("📈 평균 Test MAE:", round(best['test_MAE'], 4))

✅ 최적 ARIMAX(p,d,q): (3.0, 1.0, 1.0)
📉 평균 Train MAE: 2.6308
📈 평균 Test MAE: 3.987


#랜덤포레스트

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운송장비lag.csv')

# 인덱스 처리 (index 컬럼 없을 경우 자동 생성)
if 'index' not in df.columns:
    df.index = pd.period_range(start='2016Q1', periods=len(df), freq='Q')
else:
    df['index'] = pd.PeriodIndex(df['index'], freq='Q')
    df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '운송장비 제조업'
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI','설비투자실적BSI','업황실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y, X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 결과 저장
# -----------------------------
train_mae_list = []
test_mae_list = []

# -----------------------------
# 5️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 랜덤포레스트 모델 학습
        rf = RandomForestRegressor(
            n_estimators=500,     # 트리 개수
            max_depth=None,      # 깊이 제한 없음
            random_state=42
        )
        rf.fit(X_train, y_train)

        # 학습 구간 예측 (Train MAE)
        train_pred = rf.predict(X_train)
        train_mae = mean_absolute_error(y_train, train_pred)
        train_mae_list.append(train_mae)

        # 테스트 구간 예측 (Test MAE)
        test_pred = rf.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_pred)
        test_mae_list.append(test_mae)

    except Exception:
        continue

# -----------------------------
# 6️⃣ 결과 요약 출력
# -----------------------------
print("✅ 랜덤포레스트 롤링 예측 결과")
print("📉 평균 Train MAE:", round(np.mean(train_mae_list), 4))
print("📈 평균 Test MAE:", round(np.mean(test_mae_list), 4))


✅ 랜덤포레스트 롤링 예측 결과
📉 평균 Train MAE: 1.5289
📈 평균 Test MAE: 4.6323


#AR

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운송장비qoq.csv')

# GDP 단일 시계열만 사용 (AR(1))
y = df['운송장비 제조업'].dropna()
y.index = pd.PeriodIndex(df.loc[y.index, 'index'], freq='Q')

# -----------------------------
# 2️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

train_mae_list = []
test_mae_list = []

# -----------------------------
# 3️⃣ 롤링 예측 실행
# -----------------------------
for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    train_data = y.loc[train_start:train_end]
    test_data = y.loc[test_end:test_end]

    if len(train_data) < train_window:
        continue

    try:
        # AR(1) 학습
        model = AutoReg(train_data, lags=1, old_names=False)
        fit = model.fit()

        # --- Train 예측 ---
        # fittedvalues는 길이가 n-1이므로 첫 번째 값 제외하고 MAE 계산
        train_mae = mean_absolute_error(train_data[1:], fit.fittedvalues)
        train_mae_list.append(train_mae)

        # --- Test 예측 (1분기 앞 예측) ---
        pred = fit.predict(start=len(train_data), end=len(train_data))
        test_mae = mean_absolute_error(test_data, pred)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 4️⃣ 결과 출력
# -----------------------------
print("✅ AR(1) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ AR(1) 롤링 예측 결과
📉 평균 Train MAE: 4.9410
📈 평균 Test MAE: 6.1921


#XGB

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운송장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '운송장비 제조업'
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI','설비투자실적BSI','업황실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

# 첫 번째 윈도우로 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X.loc[start_period:first_train_end]
first_y_train = y.loc[start_period:first_train_end]

# RandomizedSearchCV 실행
xgb = XGBRegressor(random_state=42, objective='reg:squarederror')
rand_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=15,          # 전체 중 15조합 탐색
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y.loc[train_start:train_end]
    y_test = y.loc[test_end:test_end]
    X_train = X.loc[train_start:train_end]
    X_test = X.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        # 고정된 최적 파라미터로 학습
        model = XGBRegressor(**best_params, random_state=42, objective='reg:squarederror')
        model.fit(X_train, y_train)

        # 예측
        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # MAE 계산
        train_mae = mean_absolute_error(y_train, train_pred)
        test_mae = mean_absolute_error(y_test, test_pred)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 XGBoost 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'subsample': 0.8, 'n_estimators': 600, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0, 'colsample_bytree': 0.8}

📊 XGBoost 롤링 예측 결과
📉 평균 Train MAE: 0.0006
📈 평균 Test MAE: 5.3718


#MLP

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운송장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '운송장비 제조업'
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI','설비투자실적BSI','업황실적BSI'
]

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# 스케일링 (NN에서는 매우 중요)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 3️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 4️⃣ 하이퍼파라미터 후보 & RandomizedSearchCV
# -----------------------------
param_grid = {
    'hidden_layer_sizes': [(32,), (64,), (32,16), (64,32), (128,64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'lbfgs'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01]
}

# 첫 번째 윈도우로 하이퍼파라미터 탐색
first_train_end = start_period + (train_years * periods_per_year - 1)
first_X_train = X_scaled.loc[start_period:first_train_end]
first_y_train = y_scaled.loc[start_period:first_train_end]

mlp = MLPRegressor(random_state=42, max_iter=1000)
rand_search = RandomizedSearchCV(
    mlp,
    param_distributions=param_grid,
    n_iter=10,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    random_state=42
)
rand_search.fit(first_X_train, first_y_train)
best_params = rand_search.best_params_
print("✅ 최적 하이퍼파라미터:", best_params)

# -----------------------------
# 5️⃣ 롤링 예측 (최적 파라미터 고정)
# -----------------------------
train_mae_list = []
test_mae_list = []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    y_train = y_scaled.loc[train_start:train_end]
    y_test = y_scaled.loc[test_end:test_end]
    X_train = X_scaled.loc[train_start:train_end]
    X_test = X_scaled.loc[test_end:test_end]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = MLPRegressor(**best_params, random_state=42, max_iter=1000)
        model.fit(X_train, y_train)

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred.reshape(-1, 1)).flatten()
        test_pred_inv = scaler_y.inverse_transform(test_pred.reshape(-1, 1)).flatten()
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1)).flatten()

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 6️⃣ 결과 요약
# -----------------------------
print("\n📊 Neural Network (MLP) 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")


✅ 최적 하이퍼파라미터: {'solver': 'adam', 'learning_rate_init': 0.001, 'hidden_layer_sizes': (32,), 'alpha': 0.01, 'activation': 'relu'}

📊 Neural Network (MLP) 롤링 예측 결과
📉 평균 Train MAE: 0.8899
📈 평균 Test MAE: 5.4083


#LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# 1️⃣ 데이터 불러오기
# -----------------------------
df = pd.read_csv('운송장비lag.csv')

# 인덱스 설정
df['index'] = pd.PeriodIndex(df['index'], freq='Q')
df = df.set_index('index')

# -----------------------------
# 2️⃣ 예측 대상(y)과 외생 변수(X)
# -----------------------------
target = '운송장비 제조업'
exog_vars = ['가동률실적BSI','수출출하지수','채산성실적BSI','내수판매실적BSI','생산실적BSI','설비투자실적BSI','업황실적BSI']

y = df[target].dropna()
X = df[exog_vars].dropna()

# y와 X 동기화
df_model = pd.concat([y, X], axis=1).dropna()
y = df_model[target]
X = df_model[exog_vars]

# -----------------------------
# 3️⃣ 스케일링
# -----------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = pd.DataFrame(scaler_X.fit_transform(X), index=X.index, columns=X.columns)
y_scaled = pd.Series(scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten(), index=y.index)

# -----------------------------
# 4️⃣ LSTM 입력 형태 변환 함수
# -----------------------------
def create_lstm_input(X, y, lookback=4):
    """
    lookback: 과거 몇 분기 데이터를 사용할지 (ex. 4 = 1년)
    """
    Xs, ys, idxs = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X.iloc[i-lookback:i].values)
        ys.append(y.iloc[i])
        idxs.append(y.index[i])
    return np.array(Xs), np.array(ys), idxs

lookback = 4
X_seq, y_seq, idxs = create_lstm_input(X_scaled, y_scaled, lookback)
y_seq = pd.Series(y_seq, index=idxs)

# -----------------------------
# 5️⃣ 롤링 윈도우 설정
# -----------------------------
periods_per_year = 4
train_years = 5
train_window = train_years * periods_per_year

start_period = pd.Period('2016Q1', freq='Q')
end_period = pd.Period('2025Q2', freq='Q')

# -----------------------------
# 6️⃣ LSTM 모델 정의 함수
# -----------------------------
def build_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(64, activation='tanh', input_shape=input_shape, return_sequences=False))
    model.add(Dropout(0.2))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mae')
    return model

# -----------------------------
# 7️⃣ 롤링 예측
# -----------------------------
train_mae_list, test_mae_list = [], []

for test_start in pd.period_range(start_period, end_period, freq='Q'):
    train_end = test_start - 1
    test_end = test_start
    train_start = train_end - (train_window - 1)

    # 윈도우 범위 맞추기
    train_mask = (np.array(idxs) >= train_start) & (np.array(idxs) <= train_end)
    test_mask = (np.array(idxs) == test_end)

    X_train, y_train = X_seq[train_mask], y_seq[train_mask]
    X_test, y_test = X_seq[test_mask], y_seq[test_mask]

    if len(y_train) < train_window or len(X_test) == 0:
        continue

    try:
        model = build_lstm((X_train.shape[1], X_train.shape[2]))
        early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
        model.fit(X_train, y_train, epochs=200, batch_size=8, verbose=0, callbacks=[early_stop])

        # 예측
        train_pred = model.predict(X_train, verbose=0)
        test_pred = model.predict(X_test, verbose=0)

        # 스케일링 복원
        train_pred_inv = scaler_y.inverse_transform(train_pred)
        test_pred_inv = scaler_y.inverse_transform(test_pred)
        y_train_inv = scaler_y.inverse_transform(y_train.values.reshape(-1, 1))
        y_test_inv = scaler_y.inverse_transform(y_test.values.reshape(-1, 1))

        # MAE 계산
        train_mae = mean_absolute_error(y_train_inv, train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, test_pred_inv)

        train_mae_list.append(train_mae)
        test_mae_list.append(test_mae)

    except Exception as e:
        print("❌ Error:", e)
        continue

# -----------------------------
# 8️⃣ 결과 요약
# -----------------------------
print("\n📊 LSTM 롤링 예측 결과")
print(f"📉 평균 Train MAE: {np.mean(train_mae_list):.4f}")
print(f"📈 평균 Test MAE: {np.mean(test_mae_list):.4f}")



📊 LSTM 롤링 예측 결과
📉 평균 Train MAE: 3.2500
📈 평균 Test MAE: 6.5750
